# Payerne Raman lidar + radiosonde vs KENDA-CH1 801/802

Compares the temperature and specific-humidity profile above Payerne from the **Raman lidar
(RALMO)**, the **radiosonde** where one was launched that hour, and experiments **801** and **802**,
separately for **`iaf`** (post-IAU analysis) and **`lff`** (+1 h first guess).

Output:

    figures/ramanlidar_<kind>_frames/ramanlidar_<kind>_<YYYYMMDDHH>.png   profiles + CLCT bias panel
    figures/ramanlidar_<kind>_bias_timeseries.png                         the CLCT bias alone

Step through the frames with the repo's viewer:

    python3 scripts/frame_viewer.py figures/ramanlidar_iaf_frames

## Observations

**Raman lidar** — `/users/daniel/work/kenda/ramanlidar/YYYYMMDD_06610_data.txt`, 30-min cadence,
`level` in m asl. Columns are bare DWH parameter ids, in the units given by the DWH profiler
parameter table:

| id | quantity | unit |
|----|----------|------|
| `3147` | temperature | K |
| `4908` (`zlietts0`) | uncertainty of temperature | K |
| `4919` | specific humidity | g/kg |
| `4906` (`zlieuss0`) | uncertainty of specific humidity | g/kg |

The two uncertainties are **absolute**, not relative — an older version of the parameter table gives
their unit as `%`. Checked against the coincident radiosonde: read as absolute, the standardised
residual has a spread of 0.93 for temperature and 1.24 for humidity, i.e. correctly scaled, whereas
read as percent it is off by 3x and 30x. 0.30 g/kg on 5.91 g/kg is also 5.1 % relative, which is
RALMO's published performance.

`4919` being *specific* humidity is convenient: ICON `QV` is specific humidity too, so the two are
directly comparable, and the radiosonde is converted the same way rather than to a mixing ratio.

**No quality screening is applied** — every reported value is kept and plotted. Only the per-field
sentinel `10000000` is turned into a missing value, since that is the file's encoding of "not
measured" rather than a judgement about quality. The reported uncertainty is carried through and
drawn as a band, so the reliability of each level is visible in the figure instead of being decided
in the reader. Two consequences worth keeping in mind while reading the plots:

- the retrieval has a noise floor of roughly 0.2-0.3 g/kg, and above ~5-6 km, or in daylight, the
  humidity is often that floor rather than a measurement; where the air is very cold this implies an
  RH far above saturation, so the RH panel can show physically impossible values.

**Radiosonde** — `/scratch/mch/jdelbeke/soundings/output/`, launches at **00 and 12 UTC only**. In
the 801/802 window 14 launches exist, of which 6 fall on an hour that also has a lidar profile.

Two working assumptions: `termin` is treated as instantaneous (RALMO integrates over ~30 min), and
the lidar runs about 10 % dry against the coincident radiosonde, uncorrected — having the sonde on
the same axes makes that offset directly visible. It is identical in both experiments, so
801-vs-802 differences are unaffected by it.

## Model column

Mean of the 6 nearest ICON-CH1 cells to the station (46.8116 N, 6.9424 E): 746375 (0.31 km), 746374,
746368, 746365, 746372, 746370, all within 1.33 km. Their mean `HSURF` is 488.6 m against the
station's 490 m. Model levels 79-80 (centres ~504 and ~484 m asl) lie below the lidar's first gate at
551 m and are not observable by it; the sonde does reach them.

In [ ]:
# ==========================================================
# CONFIG + IMPORTS
# ==========================================================
import os

# ecCodes prints a definitions-version banner on every GRIB file it opens in this uenv. Must be set
# before earthkit.data is imported.
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

# eccodes must be imported before earthkit.data so the native libraries resolve correctly
import eccodes  # noqa: F401,E402
import earthkit.data as ekd  # noqa: E402

import collections  # noqa: E402
import warnings  # noqa: E402
from concurrent.futures import ProcessPoolExecutor  # noqa: E402
from pathlib import Path  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import xarray as xr  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import matplotlib.dates as mdates  # noqa: E402

# ----------------------------------------------------------
# Raman lidar (RALMO)
# ----------------------------------------------------------
LIDAR_DIR = Path("/users/daniel/work/kenda/ramanlidar")
STATION   = "06610"
MISSING   = 10_000_000     # per-field sentinel: "not measured", not a quality flag

# file column order: 4919 qv [g/kg], 4906 qv uncertainty [g/kg], 3147 T [K], 4908 T uncertainty [K]
LIDAR_COLS = ["wmo_id", "termin", "lat", "lon", "elev", "stn_name", "int_ind",
              "track_type", "prof_type", "type", "level",
              "qv", "qv_unc", "T", "T_unc"]

# Largest gap the level-matching fallback may interpolate across. np.inf = no limit, so every model
# level between the lowest and highest reported observation gets a value and the profile is drawn as
# one unbroken line. A finite value (150 m was used earlier) leaves a level empty where the
# instrument reported nothing near it, which breaks the line at every data hole.
MAX_INTERP_GAP_M = np.inf   # m

# ----------------------------------------------------------
# Radiosonde — same DWH export layout, different parameter ids
# ----------------------------------------------------------
SOUNDING_DIR = Path("/scratch/mch/jdelbeke/soundings/output")

# 745 T [degC], 747 dew point [degC], 748 wind speed, 743 wind direction,
# 742 height [m asl], 744 pressure [hPa]
SND_COLS = ["wmo_id", "termin", "lat", "lon", "elev", "stn_name", "int_ind", "nat_abbr",
            "track_type", "prof_type", "type", "level",
            "T_C", "Td_C", "ff", "dd", "height", "p"]

# ----------------------------------------------------------
# Model
# ----------------------------------------------------------
EXPS     = ["801", "802"]
EXP_ROOT = Path("/store_new/mch/msopr/jdelbeke/ICON_TST")

# file kind -> (subdirectory, filename prefix). Both carry T/QV/P (80 lev) and HHL (81).
KINDS = {
    "iaf": ("ANA25/det", "iaf"),   # post-IAU analysis
    "lff": ("FG25/det",  "lff"),   # +1 h first guess from the previous cycle
}

PAYERNE_CELLS = np.array([746375, 746374, 746368, 746365, 746372, 746370])
STATION_ELEV  = 490.0   # m asl
MODEL_VARS    = ["T", "QV", "P"]
N_FULL_LEV    = 80

# ----------------------------------------------------------
# Output and plotting
# ----------------------------------------------------------
CACHE_DIR  = Path("/scratch/mch/jdelbeke")
FIG_DIR    = Path("figures")
PLOT_TOP_M = 5000.0          # top of the plotted profile range, m above model ground
BAND_ALPHA = 0.30            # opacity of the observation uncertainty band

# Same palette as CLCT_spindown_decomposition.ipynb, clc_section_frames.ipynb and
# payerne_sounding_profiles.ipynb: Okabe-Ito blue and vermillion, colourblind-safe.
EXP_COLOUR = {"801": "#0072B2", "802": "#D55E00"}
SND_COLOUR = "#009E73"          # Okabe-Ito bluish green, for the radiosonde

CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)

# netCDF4 in this uenv is built against a different numpy ABI; the import warning is benign.
warnings.filterwarnings("ignore", message="numpy.ndarray size changed")

## 1. Observation readers

Both sources use the same pipe-delimited DWH layout: header on line 2, data from line 4, `10000000`
as a **per-field** sentinel — a temperature can be present where the humidity is absent, so the
sentinel is replaced per column and no rows are dropped.

Two traps in the sounding files: the `level == -100` row is a per-launch header carrying a fake
85 m height and 1000 hPa pressure with every measured field set to the sentinel, and there is no
humidity column, so specific humidity is derived from the dew point. The sonde reader returns the
same columns as the lidar reader, so one level-matching operator serves both.

In [ ]:
# ==========================================================
# Observation readers
# ==========================================================
def read_lidar_day(day):
    """One daily RALMO file. `day` is 'YYYYMMDD'.

    Everything reported is kept; only the sentinel becomes missing. Uncertainties are absolute
    (K and g/kg).
    """
    path = LIDAR_DIR / f"{day}_{STATION}_data.txt"
    if not path.exists():
        return None
    df = pd.read_csv(path, sep="|", skiprows=3, header=None, names=LIDAR_COLS, engine="c")
    if df.empty:                      # header-only: no data, or a failed DWH retrieval
        return None

    df = df.replace(MISSING, np.nan)
    df["time"] = pd.to_datetime(df["termin"].astype("int64").astype(str), format="%Y%m%d%H%M%S")
    return (df[["time", "level", "T", "T_unc", "qv", "qv_unc"]]
            .sort_values(["time", "level"]))


def load_lidar(days):
    parts = [d for d in (read_lidar_day(day) for day in days) if d is not None]
    return pd.concat(parts, ignore_index=True) if parts else None


def read_sounding_day(day):
    """One daily Payerne radiosonde file (00 and 12 UTC launches stacked).

    Returns time / level [m asl] / T [K] / qv [g/kg] / p [hPa], matching the lidar reader.
    """
    path = SOUNDING_DIR / f"{day}_{STATION}_data.txt"
    if not path.exists():
        return None
    df = pd.read_csv(path, sep="|", skiprows=3, header=None, names=SND_COLS, engine="c")
    if df.empty:
        return None
    df = df.replace(MISSING, np.nan)

    df = df[df["level"] >= 0]                    # drop the per-launch header row
    df = df[np.isfinite(df["height"])]
    if df.empty:
        return None

    df["time"] = pd.to_datetime(df["termin"].astype("int64").astype(str), format="%Y%m%d%H%M%S")
    df["T"] = df["T_C"] + 273.15

    # Saturation vapour pressure at the dew point is the actual vapour pressure. Converted to
    # SPECIFIC humidity, q = 0.622e/(p-0.378e), to match 4919 and ICON QV rather than to a mixing
    # ratio. Same saturation formula as everywhere else, so no formula mismatch can look like a
    # model error. The hygrometer drops out where the thermometer still reports.
    e = saturation_vapour_pressure(df["Td_C"] + 273.15)
    df["qv"] = 1000.0 * 0.622 * e / (df["p"] - 0.378 * e)

    return (df[["time", "height", "T", "qv", "p"]]
            .rename(columns={"height": "level"})
            .sort_values(["time", "level"]))


def load_soundings(days):
    parts = [d for d in (read_sounding_day(day) for day in days) if d is not None]
    return pd.concat(parts, ignore_index=True) if parts else None

## 2. Model column extraction

Each file is indexed once, then only the 6 target cells are read out of each level. `HHL` has to be
read rather than assumed, because ICON's levels follow the terrain. Note that
`fieldlist.sel(shortName=...)` returns an empty result on these files, so the `shortName -> [Field]`
mapping is built by hand.

In [ ]:
# ==========================================================
# Model column extraction
# ==========================================================
def model_path(kind, exp, t):
    subdir, prefix = KINDS[kind]
    return EXP_ROOT / exp / subdir / f"{prefix}{t:%Y%m%d%H}"


def extract_column(path, cells=PAYERNE_CELLS, varnames=MODEL_VARS, with_geometry=False):
    """Mean over `cells` of each level. Keeps ICON ordering: index 0 = level 1 = model top."""
    wanted = set(varnames) | ({"HHL", "HSURF"} if with_geometry else set())

    by = collections.defaultdict(list)
    for f in ekd.from_source("file", str(path)).to_fieldlist():
        sn = f.metadata("shortName")
        if sn in wanted:
            by[sn].append(f)

    out = {}
    for name in varnames:
        fields = sorted(by[name], key=lambda f: f.metadata("level"))
        if len(fields) != N_FULL_LEV:
            raise ValueError(f"{name}: expected {N_FULL_LEV} levels in {path}, got {len(fields)}")
        out[name] = np.array([f.to_numpy().ravel()[cells].mean() for f in fields])

    if with_geometry:
        hhl_f = sorted(by["HHL"], key=lambda f: f.metadata("level"))
        out["HHL"]   = np.array([f.to_numpy().ravel()[cells].mean() for f in hhl_f])
        out["HSURF"] = float(by["HSURF"][0].to_numpy().ravel()[cells].mean())
    return out


def layer_edges(hhl):
    """Full-level centres and layer top/bottom edges [m asl] from the 81 half levels."""
    top, bot = hhl[:-1], hhl[1:]      # HHL level 1 = model top, level 81 = ground
    return 0.5 * (top + bot), top, bot

## 3. Putting the observations on the model levels

Average the observation levels that fall inside a model layer; a model level is a layer mean, so
averaging is the right operator once the observations resolve the layer. If none fall inside,
interpolate to the layer centre, but only across a gap of at most `MAX_INTERP_GAP_M`.

`MAX_INTERP_GAP_M` is `np.inf`, so the fallback interpolates across any gap and each profile is drawn
as one unbroken line between the lowest and highest reported level. That keeps the figures simple to
read and to explain, at the price of levels inside a data hole carrying an interpolated value rather
than a measurement: over the 81 iaf slots, 437 temperature values per pass come from interpolation,
across a median gap of 3840 m. Below 5 km the effect is small — a finite 150 m limit changed only
2.9 % of temperature and 0.2 % of humidity cells there — but higher up the temperature line can span
kilometres of missing profile. Set the constant to a finite number of metres to break the line at
holes wider than that instead.

Relative humidity is computed over water on every curve so the lines are comparable; absolute values
below freezing are therefore not the WMO ice convention. The lidar has no pressure channel, so its
RH borrows the model's pressure, while the sonde uses its own.

In [ ]:
# ==========================================================
# Observation -> model level operator, and humidity helpers
# ==========================================================
def obs_to_model_levels(obs_level, obs_value, centres, top, bot):
    """Map one observed profile onto the model full levels. Returns (values, n_obs_in_layer)."""
    ok = np.isfinite(obs_value) & np.isfinite(obs_level)
    lev = np.asarray(obs_level, dtype=float)[ok]
    val = np.asarray(obs_value, dtype=float)[ok]

    out = np.full(len(centres), np.nan)
    cnt = np.zeros(len(centres), dtype=int)
    if lev.size == 0:
        return out, cnt

    order = np.argsort(lev)
    lev, val = lev[order], val[order]

    for k in range(len(centres)):
        inside = (lev >= bot[k]) & (lev < top[k])
        n = int(inside.sum())
        cnt[k] = n
        if n:
            out[k] = val[inside].mean()
            continue
        c = centres[k]
        below, above = lev[lev <= c], lev[lev >= c]
        if below.size and above.size \
                and (c - below[-1]) <= MAX_INTERP_GAP_M \
                and (above[0] - c) <= MAX_INTERP_GAP_M:
            out[k] = np.interp(c, lev, val)
    return out, cnt


def saturation_vapour_pressure(T_K):
    """Magnus, over water, hPa."""
    Tc = np.asarray(T_K) - 273.15
    return 6.112 * np.exp(17.62 * Tc / (243.12 + Tc))


def relative_humidity(T_K, q_gkg, p_hPa):
    """RH [%] from temperature, specific humidity [g/kg] and pressure [hPa]."""
    q = np.asarray(q_gkg) / 1000.0
    e = q * np.asarray(p_hPa) / (0.622 + 0.378 * q)
    return 100.0 * e / saturation_vapour_pressure(T_K)

## 4. Build the matched datasets

One dataset per file kind, cached to netCDF. Only hours with a lidar profile on the same hour are
kept. The cache records a provenance string covering everything that changes its contents, and is
rebuilt automatically when that changes.

Reading a file is dominated by waiting on `/store_new`, so the model extraction is spread over
worker processes; the observation side is cheap and stays in this one. Expect ~5 min per kind on the
first run and seconds afterwards.

In [ ]:
# ==========================================================
# Build (or load) the matched datasets
# ==========================================================
FORCE_REBUILD = False
MAX_TIMES     = None        # e.g. 5 for a quick test run
N_WORKERS     = min(16, os.cpu_count() or 1)


def provenance_tag():
    """Everything that changes the contents of a cache. N_WORKERS is absent: speed, not results."""
    return (f"exps={','.join(EXPS)}; cells={','.join(map(str, PAYERNE_CELLS))}; "
            f"qc=none; interp_gap<={MAX_INTERP_GAP_M}; unc=absolute; sonde=q_specific")


def available_times(kind, exp):
    subdir, prefix = KINDS[kind]
    d = EXP_ROOT / exp / subdir
    return sorted(pd.to_datetime(p.name[len(prefix):], format="%Y%m%d%H")
                  for p in d.glob(f"{prefix}" + "?" * 10))


def _slot_job(job):
    """One time slot, both experiments. Top level so ProcessPoolExecutor can pickle it."""
    kind, exps, t = job
    return {exp: extract_column(model_path(kind, exp, t)) for exp in exps}


def build_matched(kind):
    times = sorted(set.intersection(*(set(available_times(kind, e)) for e in EXPS)))
    days = sorted({t.strftime("%Y%m%d") for t in times})

    obs = load_lidar(days)
    if obs is None:
        raise RuntimeError(f"no lidar data overlapping the {kind} window")
    snd = load_soundings(days)          # may be None; soundings are only at 00 and 12 UTC

    have = set(obs["time"].unique())    # instantaneous assumption: exact match on the hour
    times = [t for t in times if np.datetime64(t) in have]
    if MAX_TIMES:
        times = times[:MAX_TIMES]
    print(f"[{kind}] {len(times)} matched hourly slots, {N_WORKERS} workers", flush=True)

    geom = extract_column(model_path(kind, EXPS[0], times[0]), varnames=[], with_geometry=True)
    centres, top, bot = layer_edges(geom["HHL"])

    nt, ne, nl = len(times), len(EXPS), N_FULL_LEV
    mod  = {v: np.full((ne, nt, nl), np.nan) for v in MODEL_VARS}
    o_T  = np.full((nt, nl), np.nan);  o_qv = np.full((nt, nl), np.nan)
    u_T  = np.full((nt, nl), np.nan);  u_qv = np.full((nt, nl), np.nan)
    s_T  = np.full((nt, nl), np.nan);  s_qv = np.full((nt, nl), np.nan)
    s_p  = np.full((nt, nl), np.nan)

    for i, t in enumerate(times):
        p = obs[obs["time"] == t]
        o_T[i],  _ = obs_to_model_levels(p["level"], p["T"],  centres, top, bot)
        o_qv[i], _ = obs_to_model_levels(p["level"], p["qv"], centres, top, bot)
        u_T[i],  _ = obs_to_model_levels(p["level"], p["T_unc"].where(p["T"].notna()),
                                         centres, top, bot)
        u_qv[i], _ = obs_to_model_levels(p["level"], p["qv_unc"].where(p["qv"].notna()),
                                         centres, top, bot)

        if snd is not None:
            g = snd[snd["time"] == t]
            if len(g):
                s_T[i],  _ = obs_to_model_levels(g["level"], g["T"],  centres, top, bot)
                s_qv[i], _ = obs_to_model_levels(g["level"], g["qv"], centres, top, bot)
                s_p[i],  _ = obs_to_model_levels(g["level"], g["p"],  centres, top, bot)

    jobs = [(kind, EXPS, t) for t in times]
    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        # pool.map preserves input order, so enumerate() still lines results up with `times`
        for i, res in enumerate(pool.map(_slot_job, jobs)):
            for j, exp in enumerate(EXPS):
                for v in MODEL_VARS:
                    mod[v][j, i] = res[exp][v]
            if (i + 1) % 20 == 0 or i == nt - 1:
                print(f"  [{kind}] {i + 1}/{nt}  {times[i]}", flush=True)

    print(f"[{kind}] slots with no T: {int((~np.isfinite(o_T)).all(axis=1).sum())}/{nt};  "
          f"no qv: {int((~np.isfinite(o_qv)).all(axis=1).sum())}/{nt};  "
          f"with a radiosonde: {int(np.isfinite(s_T).any(axis=1).sum())}/{nt}")

    return xr.Dataset(
        {
            "T_mod":  (("exp", "time", "level"), mod["T"]),
            "qv_mod": (("exp", "time", "level"), mod["QV"] * 1000.0),   # kg/kg -> g/kg
            "p_mod":  (("exp", "time", "level"), mod["P"] / 100.0),     # Pa -> hPa
            "T_obs":  (("time", "level"), o_T),
            "qv_obs": (("time", "level"), o_qv),
            "T_obs_unc":  (("time", "level"), u_T),
            "qv_obs_unc": (("time", "level"), u_qv),
            "T_snd":  (("time", "level"), s_T),
            "qv_snd": (("time", "level"), s_qv),
            "p_snd":  (("time", "level"), s_p),
            "height":     ("level", centres),
            "height_agl": ("level", centres - geom["HSURF"]),
        },
        coords={"exp": EXPS, "time": times, "level": np.arange(1, nl + 1)},
        attrs={"kind": kind, "hsurf_model_m": geom["HSURF"], "station_elev_m": STATION_ELEV,
               "provenance": provenance_tag(),
               "note": "no quality screening; lidar uncertainties are absolute (K, g/kg); "
                       "obs assumed instantaneous at termin"},
    )


DS = {}
for kind in KINDS:
    cache = CACHE_DIR / f"ramanlidar_{kind}_matched.nc"

    reuse = None
    if cache.exists() and not FORCE_REBUILD:
        candidate = xr.open_dataset(cache)
        if candidate.attrs.get("provenance") == provenance_tag():
            reuse = candidate
        else:
            candidate.close()
            print(f"[{kind}] cache is stale, rebuilding")

    if reuse is not None:
        DS[kind] = reuse
        print(f"[{kind}] loaded cache ({DS[kind].sizes['time']} slots): {cache}")
    else:
        DS[kind] = build_matched(kind)
        DS[kind].to_netcdf(cache)
        print(f"[{kind}] wrote cache ({DS[kind].sizes['time']} slots): {cache}")

## 5. The verification bias time series

The panel under every frame, taken from MOVERO exactly as in `clc_section_frames.ipynb`, and matched
to the file kind being plotted: the **`lff` frames carry the +1 h first-guess CLCT bias**
(`time_scores01_CLCT.dat` under `801-FG-det_<subset>`), the **`iaf` frames carry the +0 h analysis
CLCT bias** (`time_scores00_CLCT.dat` under `801-K-CH1-det_<subset>`). `ME` is the bias in octa.

`time_scores*_CLCT.dat` is an ATAB file with a variable-length header, so the column names are taken
from the row starting with `YYYY` rather than a fixed line number, and the file's own sentinel
(-0.9999e9) becomes NaN.

Two things this data cannot do, worth keeping in view while watching the frames:

- **No lead-time axis.** Every row in a given file has one lead time, so the spin-up signature —
  error evolving over forecast hours 0-6 — is not in these files. What moves along the panel is the
  bias in calendar time.
- **Different geometry from the profiles above.** The bias is over Swiss station points; the panels
  above are one column over Payerne. They answer related but not identical questions, so a mismatch
  between them is not a contradiction.

In [ ]:
# ==========================================================
# MOVERO CLCT bias, per file kind
# ==========================================================
MOVERO_WD_FG  = Path("/scratch/mch/jdelbeke/movero/wd/2025_kenda_801_802")
MOVERO_WD_ANA = Path("/scratch/mch/jdelbeke/movero/wd/2025_kenda_801_802_iaf")

VERIF_SUBSET = "ch-sp"     # station subset: ch (all), ch-am, ch-sp, ch-av
SUBSET_LABEL = {"ch": "all Swiss stations", "ch-am": "Alps", "ch-sp": "Swiss plateau",
                "ch-av": "Alpine valleys"}
VERIF_PARAM = "CLCT"
VERIF_SCORE = "ME"         # mean error = bias, in octa

# file kind -> (movero working directory, version-directory template, lead-time range).
# lff is the +1 h first guess, iaf the +0 h initialised analysis, so each takes its own scores.
KIND_VERIF = {
    "lff": (MOVERO_WD_FG,  "{exp}-FG-det_{subset}",    "01"),
    "iaf": (MOVERO_WD_ANA, "{exp}-K-CH1-det_{subset}", "00"),
}


def read_movero_time_scores(exp_dir, ltr, param=VERIF_PARAM):
    """One MOVERO ATAB time_scores file as a DataFrame indexed by valid time."""
    path = Path(exp_dir) / f"time_scores{ltr}_{param}.dat"
    with open(path) as fh:
        lines = fh.readlines()
    i_hdr = next(i for i, line in enumerate(lines) if line.split()[:1] == ["YYYY"])
    names = lines[i_hdr].split()
    df = pd.read_csv(path, sep=r"\s+", skiprows=i_hdr + 1, names=names, header=None)
    df = df.mask(df < -1e8)                     # the missing-value sentinel
    df["time"] = pd.to_datetime(dict(year=df["YYYY"], month=df["MM"], day=df["DD"],
                                     hour=df["hh"], minute=df["mm"]))
    return df.set_index("time").sort_index()


BIAS_DF, BIAS = {}, {}
for kind, (wd, tmpl, ltr) in KIND_VERIF.items():
    BIAS_DF[kind] = {e: read_movero_time_scores(wd / tmpl.format(exp=e, subset=VERIF_SUBSET), ltr)
                     for e in EXPS}
    BIAS[kind] = {e: df[VERIF_SCORE] for e, df in BIAS_DF[kind].items()}
    for e, df in BIAS_DF[kind].items():
        lt = sorted(df["lt_hh"].dropna().unique())
        print(f"[{kind}] {e}: {len(df)} rows, {df.index[0]:%d %b %H} - {df.index[-1]:%d %b %H} UTC, "
              f"lead time {lt} h, {int(df['N'].max())} stations at most, "
              f"{VERIF_SCORE} mean {BIAS[kind][e].mean():+.3f} octa "
              f"({BIAS[kind][e].notna().sum()} valid hours)")

# Fixed across every frame and both kinds: the marker has to be comparable from one PNG to the next.
BIAS_TSPAN = (min(b.index[0] for k in BIAS for b in BIAS[k].values()),
              max(b.index[-1] for k in BIAS for b in BIAS[k].values()))
_all = np.concatenate([b.dropna().values for k in BIAS for b in BIAS[k].values()])
_pad = 0.12 * (np.nanmax(_all) - np.nanmin(_all))
BIAS_YLIM = (np.nanmin(_all) - _pad, np.nanmax(_all) + _pad)
print(f"\nfrozen panel: {BIAS_TSPAN[0]:%d %b %H} - {BIAS_TSPAN[1]:%d %b %H} UTC, "
      f"{VERIF_SCORE} {BIAS_YLIM[0]:+.2f} to {BIAS_YLIM[1]:+.2f} octa")


def draw_bias_panel(ax, kind, t=None, legend=True):
    """The bias timeseries for one file kind, optionally with the marker for hour `t`.

    Axis limits are the frozen ones, not data-driven, so the marker sits at the same place for the
    same time in every frame.
    """
    for e in EXPS:
        b = BIAS[kind][e]
        ax.plot(b.index, b.values, "-", color=EXP_COLOUR[e], lw=1.3,
                label=f"{e}  {VERIF_PARAM} {VERIF_SCORE}")
    ax.axhline(0, color="k", lw=0.9)
    ax.set_xlim(*BIAS_TSPAN)
    ax.set_ylim(*BIAS_YLIM)
    ax.grid(alpha=0.3)
    ax.set_ylabel(f"{VERIF_PARAM} bias [octa]")
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=range(0, 24, 6)))
    if legend:
        ax.legend(fontsize=8, ncol=len(EXPS), loc="upper right")

    if t is None:
        return
    ts = pd.Timestamp(t)
    ax.axvline(ts, color="k", lw=1.4, ls=":", zorder=4)
    shown = []
    for e in EXPS:
        val = BIAS[kind][e].get(ts, np.nan)
        if np.isfinite(val):
            ax.plot([ts], [val], marker="*", ms=26, color=EXP_COLOUR[e],
                    mec="k", mew=0.9, zorder=6, clip_on=False)
            shown.append(f"{e} {val:+.2f}")
    ax.set_title(f"{kind} {VERIF_PARAM} bias, "
                 f"{SUBSET_LABEL.get(VERIF_SUBSET, VERIF_SUBSET)}; "
                 + ("now: " + ",  ".join(shown) + " octa" if shown
                    else "no verification at this hour"),
                 fontsize=10, loc="left")


# Each series on its own, for reference; every frame of that kind carries the same panel.
for kind in KINDS:
    fig, ax = plt.subplots(figsize=(13, 4), constrained_layout=True)
    draw_bias_panel(ax, kind)
    ax.set_title(f"{kind} {VERIF_PARAM} bias, "
                 f"{SUBSET_LABEL.get(VERIF_SUBSET, VERIF_SUBSET)} ({VERIF_SUBSET}), "
                 f"{VERIF_SCORE} in octa", fontsize=11, loc="left")
    ax.set_xlabel("valid time [UTC]")
    out = FIG_DIR / f"ramanlidar_{kind}_bias_timeseries.png"
    fig.savefig(out, dpi=130)
    plt.show()
    print(f"[{kind}] -> {out}")

## 6. Profile frames

One PNG per valid time and file kind. The three panels on top are the profiles — RALMO in black with
its reported uncertainty (K, g/kg) as a band, the radiosonde green dashed where one exists, 801 blue
and 802 vermillion. The band is exactly `value +/- reported uncertainty`, raw and unscaled - nothing
is clipped, rescaled or corrected, so where the reported uncertainty is large the band is large. Underneath, the MOVERO CLCT bias for that file kind spans the full width, with a dotted
line and a star marking the hour drawn above, so every frame shows where it sits in the sequence.

Since nothing is screened out, the humidity band fills the whole panel wherever the reported
uncertainty exceeds the plotted range — in daylight above ~2 km that is common, and it is the
clearest signal in the figure that the retrieval there is noise rather than measurement.

Profile axis limits are fixed across every frame and shared between `iaf` and `lff`, taken from the
0.5-99.5 percentile of everything plotted below `PLOT_TOP_M`. That keeps noise-dominated levels from
stretching the axes; individual points can therefore run off-scale rather than being removed. Frames
with no lidar value at all are skipped and counted.

In [ ]:
# ==========================================================
# Profile frames, each carrying the CLCT bias panel with a marker
# ==========================================================
OVERWRITE_FRAMES = False        # True to redraw frames that already exist


def snd_rh(ds, i=slice(None)):
    """Sounding RH, using the sonde's own pressure."""
    return relative_humidity(ds["T_snd"].values[i], ds["qv_snd"].values[i], ds["p_snd"].values[i])


def obs_rh(ds, i=slice(None)):
    """Lidar RH, borrowing the model pressure (the lidar has no pressure channel)."""
    p = ds["p_mod"].values[0] if isinstance(i, slice) else ds["p_mod"].values[0, i]
    return relative_humidity(ds["T_obs"].values[i], ds["qv_obs"].values[i], p)


def frame_limits(datasets):
    """Fixed profile axis limits, shared across every frame and both kinds."""
    def collect(pick):
        vals = []
        for ds in datasets:
            z_ok = ds["height_agl"].values <= PLOT_TOP_M
            for a in pick(ds):
                vals.append(np.asarray(a)[..., z_ok].ravel())
        v = np.concatenate(vals)
        return v[np.isfinite(v)]

    def rng(pick, pad_frac=0.05, floor=None):
        v = collect(pick)
        lo, hi = np.percentile(v, [0.5, 99.5])
        pad = pad_frac * (hi - lo)
        lo, hi = lo - pad, hi + pad
        return (max(lo, floor) if floor is not None else lo), hi

    def rh_of(ds):
        out = [relative_humidity(ds["T_mod"].values[j], ds["qv_mod"].values[j],
                                 ds["p_mod"].values[j]) for j in range(ds.sizes["exp"])]
        return out + [obs_rh(ds), snd_rh(ds)]

    return {
        "T":  rng(lambda d: [d["T_mod"].values, d["T_obs"].values, d["T_snd"].values]),
        "qv": rng(lambda d: [d["qv_mod"].values, d["qv_obs"].values, d["qv_snd"].values],
                  floor=0.0),
        "rh": rng(rh_of, floor=0.0),
    }


def draw_frame(ds, i, lims, kind, outpath):
    z = ds["height_agl"].values
    t = pd.Timestamp(ds["time"].values[i])

    fig = plt.figure(figsize=(14, 9.5), constrained_layout=True)
    # one extra row, shorter than the profiles, spanning all three columns
    gs = fig.add_gridspec(2, 3, height_ratios=[3.0, 1.0])

    # sharey by hand: add_gridspec has no grid-wide sharey, and the bias panel must stay out of
    # the sharing - its axes are time and octa, not g/kg and metres
    ax = [fig.add_subplot(gs[0, 0])]
    ax += [fig.add_subplot(gs[0, c], sharey=ax[0]) for c in (1, 2)]
    for a in ax[1:]:
        a.tick_params(labelleft=False)
    ax_bias = fig.add_subplot(gs[1, :])

    for a, var in ((ax[0], "T_obs"), (ax[1], "qv_obs")):
        v, u = ds[var].values[i], ds[f"{var}_unc"].values[i]
        a.fill_betweenx(z, v - u, v + u, color="0.6", alpha=BAND_ALPHA,
                        lw=0.5, edgecolor="0.45", zorder=2)
        a.plot(v, z, "k.-", ms=3, lw=1.2, label="RALMO", zorder=4)
    ax[2].plot(obs_rh(ds, i), z, "k.-", ms=3, lw=1.2, label="RALMO (+ model p)", zorder=4)

    has_snd = bool(np.isfinite(ds["T_snd"].values[i]).any()
                   or np.isfinite(ds["qv_snd"].values[i]).any())
    if has_snd:
        style = dict(color=SND_COLOUR, ls="--", lw=1.5, zorder=3)
        ax[0].plot(ds["T_snd"].values[i],  z, label="sounding", **style)
        ax[1].plot(ds["qv_snd"].values[i], z, label="sounding", **style)
        ax[2].plot(snd_rh(ds, i), z, label="sounding (own p)", **style)

    for j, exp in enumerate(ds["exp"].values):
        c = EXP_COLOUR.get(str(exp), f"C{j}")
        ax[0].plot(ds["T_mod"].values[j, i],  z, "-", color=c, lw=1.4, label=exp)
        ax[1].plot(ds["qv_mod"].values[j, i], z, "-", color=c, lw=1.4, label=exp)
        ax[2].plot(relative_humidity(ds["T_mod"].values[j, i], ds["qv_mod"].values[j, i],
                                     ds["p_mod"].values[j, i]), z, "-", color=c, lw=1.4, label=exp)

    ax[2].axvline(100, color="0.6", lw=0.8, zorder=1)

    ax[0].set_xlabel("T [K]");             ax[0].set_xlim(*lims["T"])
    ax[1].set_xlabel("qv [g/kg]");         ax[1].set_xlim(*lims["qv"])
    ax[2].set_xlabel("RH over water [%]"); ax[2].set_xlim(*lims["rh"])
    ax[0].set_ylabel("height above model ground [m]")
    ax[0].set_ylim(0, PLOT_TOP_M)          # explicit, not invert_yaxis(), which toggles
    for a in ax:
        a.grid(alpha=0.25)
        a.legend(fontsize=8, loc="upper right")

    draw_bias_panel(ax_bias, kind, t)
    ax_bias.set_xlabel("valid time [UTC]   |   star = the hour drawn above")

    n_T  = int(np.isfinite(ds["T_obs"].values[i]).sum())
    n_qv = int(np.isfinite(ds["qv_obs"].values[i]).sum())
    fig.suptitle(f"Payerne RALMO{' + sounding' if has_snd else ''} vs {kind}"
                 f"  -  {t:%Y-%m-%d %H:%M} UTC   (RALMO levels: T {n_T}, qv {n_qv})")
    fig.savefig(outpath, dpi=110)
    plt.close(fig)


LIMS = frame_limits(list(DS.values()))
FRAME_TAG = (provenance_tag() + f"; limits={LIMS}; top={PLOT_TOP_M}; band_alpha={BAND_ALPHA}"
             f"; palette={EXP_COLOUR},{SND_COLOUR}"
             f"; verif={VERIF_PARAM}/{VERIF_SCORE}/{VERIF_SUBSET}"
             f"; bias_ylim={BIAS_YLIM}; bias_tspan={BIAS_TSPAN}")

for kind, ds in DS.items():
    outdir = FIG_DIR / f"ramanlidar_{kind}_frames"
    outdir.mkdir(parents=True, exist_ok=True)

    tagfile = outdir / "provenance.txt"
    stale = tagfile.read_text() != FRAME_TAG if tagfile.exists() else any(outdir.glob("*.png"))

    written = skipped_empty = skipped_exist = with_snd = 0
    for i in range(ds.sizes["time"]):
        if not (np.isfinite(ds["T_obs"].values[i]).any()
                or np.isfinite(ds["qv_obs"].values[i]).any()):
            skipped_empty += 1
            continue
        t = pd.Timestamp(ds["time"].values[i])
        out = outdir / f"ramanlidar_{kind}_{t:%Y%m%d%H}.png"
        if np.isfinite(ds["T_snd"].values[i]).any():
            with_snd += 1
        if out.exists() and not (OVERWRITE_FRAMES or stale):
            skipped_exist += 1
            continue
        draw_frame(ds, i, LIMS, kind, out)
        written += 1

    tagfile.write_text(FRAME_TAG)
    print(f"[{kind}] {written} written, {skipped_exist} already present, "
          f"{skipped_empty} skipped (no observation), {with_snd} with a radiosonde  ->  {outdir}")

## 7. Stepping through the frames

```bash
python3 scripts/frame_viewer.py figures/ramanlidar_iaf_frames
python3 scripts/frame_viewer.py figures/ramanlidar_lff_frames
```

Arrow keys step, the slider scrubs, `play` animates. Stdlib only, so the plain login-node `python3`
runs it. Frame names sort chronologically, so they also feed straight into `ffmpeg`.